# Task 2 - Exploratory Data Analysis (EDA)

This notebook performs a complete exploratory data analysis on the UCI Adult dataset. The goal is to inspect, clean, analyze, and visualize the dataset before summarizing findings and conclusions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
    'hours-per-week', 'native-country', 'income'
]

df = pd.read_csv(url, header=None, names=columns, na_values='?')
df.head()


## 1. Initial Dataset Inspection

We inspect the shape, columns, data types, missing values, and duplicates before any cleaning.


In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\nColumn names:\n", df.columns.tolist())
print("\nData types:\n")
print(df.dtypes)
print("\nMissing values:\n")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


## 2. Data Cleaning

This section handles missing values, duplicates, and common inconsistencies.


In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Fill missing categorical values with the mode
for col in ['workclass', 'occupation', 'native-country']:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# Convert the income target to a binary label
df['income'] = df['income'].str.strip()
df['income_binary'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

# Convert numeric columns to numeric types
numeric_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Clean any remaining missing numeric values
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

df.info()
df.head()


## 3. Univariate Analysis

We examine important numeric and categorical variables individually.


In [ ]:
# Numeric statistics
numeric_summary = df[numeric_cols].describe().T
print(numeric_summary)

# Age distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['age'], bins=25, kde=True)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

# Income distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='income', order=['<=50K', '>50K'])
plt.title('Income Distribution')
plt.xlabel('Income Category')
plt.ylabel('Count')
plt.show()

# Education distribution
plt.figure(figsize=(12, 5))
sns.countplot(data=df, x='education', order=df['education'].value_counts().index)
plt.xticks(rotation=45)
plt.title('Education Distribution')
plt.xlabel('Education Level')
plt.ylabel('Count')
plt.show()


## 4. Bivariate Analysis

We analyze relationships between important variables such as income, age, education, and work hours.


In [ ]:
# Income vs age
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='income', y='age')
plt.title('Income by Age')
plt.xlabel('Income Category')
plt.ylabel('Age')
plt.show()

# Education and income
plt.figure(figsize=(14, 6))
education_income = df.groupby(['education', 'income']).size().unstack(fill_value=0)
education_income.plot(kind='bar', stacked=True)
plt.title('Income by Education Level')
plt.xlabel('Education')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

# Hours per week and income
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='income', y='hours-per-week')
plt.title('Income by Weekly Work Hours')
plt.xlabel('Income Category')
plt.ylabel('Hours per Week')
plt.show()


## 5. Outlier Detection

We use the IQR method to identify outliers in numeric variables.


In [ ]:
def detect_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

for col in numeric_cols:
    n_outliers = detect_outliers_iqr(df[col])
    print(f"{col}: {n_outliers} potential outliers")

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, y='capital-gain')
plt.title('Capital Gain Outlier Check')
plt.ylabel('Capital Gain')
plt.show()


## 6. Correlation Analysis

The numerical variables are checked for relationships using a correlation matrix.


In [ ]:
corr = df[numeric_cols + ['income_binary']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

print(corr[['income_binary']].sort_values('income_binary', ascending=False))


## 7. Key Insights

At least seven meaningful insights are summarized below.

1. The median age is close to the middle of the working-age population, with a right-skewed age distribution.
2. The majority of individuals earn <=50K, while the high-income group is a minority.
3. Higher education levels tend to be associated with greater representation in the >50K income group.
4. Individuals with higher weekly work hours are more likely to fall into the >50K income category.
5. Capital gain and capital loss have strong dispersion, suggesting outlier-driven wealth differences.
6. Correlation results show that education-num and age have practical relationships with income status.
7. The dataset contains missing values in categorical fields, which must be addressed through imputation rather than deletion to retain data volume.
8. The income distribution is not balanced, which is important for model evaluation if a prediction model is later built.


In [ ]:
# Save cleaned data
df.to_csv('adult_cleaned.csv', index=False)
print('Cleaned dataset saved to adult_cleaned.csv')
